# 23.2 GCP 数据科学栈:BigQuery 与按量计费 / GCP for Data Science: BigQuery & Pay-Per-Query

**中文**:Google Cloud(GCP)在数据科学圈有一个杀手级产品:**BigQuery** —— 一个**完全无服务器的数据仓库**。它颠覆性在于:你**不用管任何集群**(不像 Spark/Redshift 要配节点),直接写 SQL,它能在**几秒内查询 PB 级数据**,按**扫描的字节数**计费。这背后是我们反复见到的思想——**存算分离 + 列式存储**的极致体现。但它的计费模型有一个必须理解的关键:**你为"扫描了多少数据"付费,而不是"返回了多少行"**。这意味着一句 `SELECT *` 可能比一句精确查询贵几十倍。本节用 DuckDB + Parquet(BigQuery 的思想同源)**真实测量**这个成本模型,让你亲眼看到"分区裁剪 + 只选需要的列"如何省 60 倍的钱,再给出 GCP 数据科学服务全景。
**English**: Google Cloud (GCP) has a killer product in data science: **BigQuery** — a **fully serverless data warehouse**. Its disruption: you **manage no clusters** (unlike Spark/Redshift needing node configuration), just write SQL, and it can **query petabyte-scale data in seconds**, billed by **bytes scanned**. Behind this is the idea we've seen repeatedly — the ultimate expression of **storage-compute separation + columnar storage**. But its billing model has a key you must understand: **you pay for "how much data was scanned," not "how many rows were returned."** This means one `SELECT *` can cost tens of times more than a precise query. This section uses DuckDB + Parquet (same idea as BigQuery) to **really measure** this cost model, letting you see how "partition pruning + selecting only needed columns" saves 60x, then gives the GCP data-science service map.

---

**中文**:**BigQuery 为什么无服务器且快**:①**存算分离**——数据列式存储在 Google 的分布式存储上,查询时**临时调用海量计算资源**并行扫描,查完释放。你永远不用配置或管理任何"集群"。②**列式 + 极致并行**——只读查询涉及的列(列裁剪),配合分区把扫描量降到最低。③**按扫描字节计费**(约 $6.25/TB)——这是理解 BigQuery 成本的**唯一关键**:成本 ∝ 查询**扫描**的字节数,和返回多少行、花多少时间都无关。
**English**: **Why BigQuery is serverless and fast**: ① **storage-compute separation** — data is stored columnar on Google's distributed storage, and queries **temporarily summon massive compute** to scan in parallel, releasing after. You never configure or manage any "cluster." ② **columnar + extreme parallelism** — read only the columns the query touches (column pruning), with partitioning minimizing the scan. ③ **billed per bytes scanned** (~$6.25/TB) — this is the **sole key** to BigQuery cost: cost ∝ the bytes a query **scans**, regardless of rows returned or time taken.

**中文**:**推论(极其重要的实践规则)**:因为按扫描量计费,**减少扫描量 = 直接省钱**:
**English**: **Corollary (a crucial practical rule)**: because you pay per bytes scanned, **reducing the scan = directly saving money**:
- **中文**:**永远不要 `SELECT *`**——它扫描所有列,包括你根本不用的巨大字段。只选你要的列。
  **Never `SELECT *`** — it scans all columns, including huge fields you don't use. Select only the columns you need.
- **中文**:**用分区(partition)+ 分区过滤**——按日期分区的表,`WHERE date='2024-01-01'` 只扫描那一天的数据,而非全表。
  **Use partitions + partition filters** — for a date-partitioned table, `WHERE date='2024-01-01'` scans only that day, not the whole table.
- **中文**:**用聚簇(clustering)** 进一步减少扫描;用**预览/`LIMIT` 不会减少扫描**(仍全扫,别用它省钱)。
  **Use clustering** to further reduce scans; note **preview/`LIMIT` does NOT reduce scanning** (still full-scans, don't use it to save money).

> 💡 **面试速查 / Interview cheat-sheet（★★ GCP/数仓成本必考）**
> **中文**:**GCP 数据科学栈**:**GCS**(对象存储=S3)、**BigQuery**(无服务器数仓, 杀手级)、**Vertex AI**(ML 平台=SageMaker)、**Dataflow**(Apache Beam, 流批统一)、**Pub/Sub**(消息=Kafka)、**Dataproc**(托管 Spark=EMR)、**Looker**(BI)。**BigQuery 核心**:①无服务器(不管集群, 秒查 PB)②存算分离+列式③**按扫描字节计费(~$6.25/TB)**——成本∝扫描量, 与返回行数无关。**省钱铁律**:①**别 SELECT ***(只选需要列)②**分区+分区过滤**(WHERE date=... 只扫该分区)③聚簇④注意 **LIMIT/预览不减扫描**⑤物化视图/BI Engine 缓存⑥容量定价 vs 按需定价。**vs 其他数仓**:BigQuery/Snowflake=无服务器/存算分离/弹性; Redshift=需管节点(较老)。**特色**:BQ ML(SQL 里训模型)、GIS、流式插入。面试金句:*"BigQuery 是无服务器数仓, 存算分离+列式让它秒查 PB 且不用管集群; 关键是按扫描字节计费——成本正比于扫描量而非返回行数, 所以省钱靠只选需要的列(别 SELECT *)、分区+分区过滤减少扫描, 注意 LIMIT 不减扫描; GCP 对应关系 GCS=S3、Vertex AI=SageMaker、Dataflow=Beam、Pub/Sub=Kafka。"*
> **English**: **GCP data-science stack**: **GCS** (object storage = S3), **BigQuery** (serverless warehouse, killer), **Vertex AI** (ML platform = SageMaker), **Dataflow** (Apache Beam, unified stream/batch), **Pub/Sub** (messaging = Kafka), **Dataproc** (managed Spark = EMR), **Looker** (BI). **BigQuery core**: ① serverless (no clusters, second-scale PB queries) ② storage-compute separation + columnar ③ **billed per bytes scanned (~$6.25/TB)** — cost ∝ scan, independent of rows returned. **Cost rules**: ① **never SELECT *** (select only needed columns) ② **partition + partition filter** (WHERE date=... scans only that partition) ③ clustering ④ note **LIMIT/preview don't reduce scanning** ⑤ materialized views/BI Engine caching ⑥ capacity vs on-demand pricing. **vs other warehouses**: BigQuery/Snowflake = serverless/storage-compute-separated/elastic; Redshift = manage nodes (older). **Specials**: BQ ML (train models in SQL), GIS, streaming inserts. Interview line: *"BigQuery is a serverless warehouse; storage-compute separation + columnar let it query PB in seconds with no clusters; the key is billing per bytes scanned — cost is proportional to the scan not rows returned, so save money by selecting only needed columns (never SELECT *) and partitioning + partition filters to reduce scanning, noting LIMIT doesn't reduce the scan; GCP mappings: GCS=S3, Vertex AI=SageMaker, Dataflow=Beam, Pub/Sub=Kafka."*


In [ ]:

# ============================================================
# 真实测量 BigQuery 的"按扫描字节计费"模型 / really measure BigQuery's "pay per bytes scanned"
# 中文:BigQuery 的思想和 Parquet 列式同源。我们造一张按日期分区的 Parquet "表"(含一个巨大的 raw_payload 列),
#      测量不同查询"要扫描多少字节"——这直接决定 BigQuery 的花费。
# English: BigQuery shares the columnar idea with Parquet. We build a date-partitioned Parquet "table" (with a huge
#      raw_payload column) and measure "how many bytes each query must scan" — which directly determines BigQuery cost.
# ============================================================
import numpy as np, pandas as pd, os, shutil
ROOT="/tmp/bq_table"
if os.path.exists(ROOT): shutil.rmtree(ROOT)
os.makedirs(ROOT)
rng=np.random.default_rng(0)
dates=["2024-01-01","2024-01-02","2024-01-03","2024-01-04"]; N=300000
for d in dates:                                            # 按日期分区(每天一个目录)/ partitioned by date
    df=pd.DataFrame({"user_id":rng.integers(0,100000,N),
                     "revenue":rng.random(N)*100,
                     "device":rng.choice(["ios","android","web"],N),
                     "raw_payload":[os.urandom(60).hex() for _ in range(N)]})   # 巨大的不可压缩列(真实事件表常有)/ huge column
    os.makedirs(f"{ROOT}/date={d}", exist_ok=True); df.to_parquet(f"{ROOT}/date={d}/data.parquet")

total_mb=sum(os.path.getsize(f"{ROOT}/date={d}/data.parquet") for d in dates)/1e6
one=pd.read_parquet(f"{ROOT}/date={dates[0]}/data.parquet")
col_mb={}                                                  # 逐列大小(=扫描该列的字节)/ per-column size = bytes scanned for that column
for c in one.columns:
    one[[c]].to_parquet("/tmp/_c.parquet"); col_mb[c]=os.path.getsize("/tmp/_c.parquet")/1e6
print(f"整张表 {total_mb:.0f} MB (4 天分区)。单分区各列大小 MB / per-column size:")
for c,mb in col_mb.items(): print(f"    {c:14} {mb:6.2f} MB")

# 三种查询"扫描的字节" / bytes scanned by three queries
scan_all      = total_mb                                   # SELECT * 全表全列 / whole table, all columns
scan_1part    = sum(col_mb.values())                       # SELECT * WHERE date=1天 → 分区裁剪 / 1 partition, all columns
scan_1col1part= col_mb["revenue"]                          # SELECT sum(revenue) WHERE date=1天 → 分区+列裁剪 / 1 partition, 1 column
PRICE=6.25/1e6                                             # $6.25/TB → $ per MB
print(f"\n{'查询 query':38}{'扫描 MB':>10}{'省多少':>10}")
print(f"{'A) SELECT * (全表全列)':38}{scan_all:>10.1f}{'1x(基准)':>10}")
print(f"{'C) SELECT * WHERE date=1天':38}{scan_1part:>10.1f}{f'{scan_all/scan_1part:.0f}x':>10}")
print(f"{'B) SELECT sum(revenue) WHERE date=1天':38}{scan_1col1part:>10.2f}{f'{scan_all/scan_1col1part:.0f}x':>10}")
print(f"\n若每天跑该查询 1000 次的费用 / cost if run 1000×/day:")
print(f"  A) SELECT *              ${scan_all*PRICE*1000:.2f}/天")
print(f"  B) 分区+列裁剪            ${scan_1col1part*PRICE*1000:.4f}/天   → 省 {scan_all/scan_1col1part:.0f} 倍!")
print("→ BigQuery 按扫描字节计费: 分区过滤 + 只选需要的列 = 同样结果, 花费差 60 倍。永远别 SELECT *")


In [ ]:

# ============================================================
# 可视化:BigQuery 成本模型 + GCP 服务地图 / BigQuery cost model + GCP service map
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
qs=["A) SELECT *\n全表全列","C) SELECT *\nWHERE 1天","B) sum(revenue)\nWHERE 1天"]
scans=[scan_all, scan_1part, scan_1col1part]
b=ax[0].bar(qs,scans,color=["#C44E52","#DD8452","#55A868"])
for bar,s in zip(b,scans): ax[0].text(bar.get_x()+bar.get_width()/2,s+2,f"{s:.1f}MB",ha="center",fontsize=10,weight="bold")
ax[0].set_ylabel("扫描字节 MB(= 花费)"); ax[0].set_title(f"BigQuery 按扫描计费:分区+列裁剪省 {scan_all/scan_1col1part:.0f}x")
# GCP vs AWS 对应地图 / GCP↔AWS mapping
ax[1].axis("off"); ax[1].set_title("GCP 服务地图(括号=AWS 对应)",fontsize=12,weight="bold")
rows=[("对象存储","GCS  (= S3)","#4C72B0"),
      ("无服务器数仓","BigQuery  (= Athena/Redshift)","#55A868"),
      ("ML 平台","Vertex AI  (= SageMaker)","#9467BD"),
      ("流批处理","Dataflow/Beam  (= Kinesis/Glue)","#DD8452"),
      ("消息队列","Pub/Sub  (= Kinesis/Kafka)","#C44E52"),
      ("托管 Spark","Dataproc  (= EMR)","#8172B3")]
for i,(cat,svc,c) in enumerate(rows):
    ax[1].add_patch(plt.Rectangle((0.05,0.83-i*0.14),0.9,0.11,fc=c,alpha=0.2,ec=c,transform=ax[1].transAxes))
    ax[1].text(0.08,0.885-i*0.14,cat,fontsize=8.5,weight="bold",color=c,transform=ax[1].transAxes)
    ax[1].text(0.45,0.885-i*0.14,svc,fontsize=8.5,transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/cloud02_viz.png",dpi=80); plt.show()
print("左:同样的结果, SELECT * 比精确查询贵 60 倍(全因扫描量); 右:GCP 服务与 AWS 一一对应")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **BigQuery 把"无服务器"推到极致:你连集群都看不见**:比起要配节点的 Redshift、要管资源的 Spark,BigQuery 的体验是——你只写一句 SQL,几秒后 PB 级结果就回来了,**你完全不知道(也不需要知道)背后调动了多少台机器**。这是存算分离 + 列式的极致:数据静静躺在存储里,查询来了就临时召唤海量计算并行扫描、查完释放。对数据科学家,这意味着**你可以专注于 SQL 和分析,而不是运维**——这正是云"免运维"价值的最佳体现。同类的还有 Snowflake(下下节),它们代表了数仓的现代形态。
2. **"按扫描字节计费"是一把双刃剑,理解它=省钱,忽视它=账单爆炸**:我们的真实测量给出了震撼的数字——同样是查"某天的总收入",一句 `SELECT *`(扫描全表 162MB)比一句 `SELECT sum(revenue) WHERE date=...`(扫描 2.7MB)**贵 60 倍**,而两者的**正确结果完全一样**。如果你的查询每天跑几千次、表有 TB 级,这个差异就是每月几千美元 vs 几十美元。关键认知:①**你为扫描的列付费**,那个你根本不看的巨大 `raw_payload` 列,`SELECT *` 会把它全扫一遍;②**你为扫描的分区付费**,不加分区过滤就是全表扫描;③**最反直觉的坑:`LIMIT 10` 不省钱**——它仍然全表扫描,只是最后只返回 10 行(计费按扫描,不按返回)。这些不是"优化技巧",而是用 BigQuery 的**基本素养**。
3. **诚实的边界:无服务器很爽,但要清醒它的取舍和适用面**。①**成本可预测性差**:按量计费意味着一个写错的查询(忘了分区过滤、或 join 炸了)可能瞬间扫描几十 TB、烧掉一大笔钱——所以生产要设**成本控制**(查询字节上限、预算告警、用容量定价锁定成本)。②**不是所有负载都适合**:BigQuery 为**大规模分析扫描(OLAP)** 优化,不适合高频的小事务/低延迟点查(那是 OLTP 数据库或 Bigtable 的活);频繁的小查询在按量模型下也可能不划算。③**厂商锁定**:深度用 BigQuery 的专有特性(BQ ML、专有函数)会绑死在 GCP,迁移成本高——所以很多团队坚持**用标准 SQL + 开放格式(把数据也存一份 Parquet/Iceberg 在 GCS)**,保留可迁移性。④**三大云高度同构**:GCS≈S3、BigQuery≈Athena/Redshift/Snowflake、Vertex AI≈SageMaker、Pub/Sub≈Kafka——**学会一朵云的心智模型,换云只是查文档改 API**,别被厂商的营销名词吓住。**结论:BigQuery 代表无服务器数仓的极致——存算分离让你秒查 PB 且免运维; 但它按扫描字节计费, 省钱的铁律是分区过滤 + 只选需要的列(别 SELECT *、别指望 LIMIT 省钱), 一个疏忽就能烧掉几十倍的钱; 要设成本护栏、警惕锁定, 并认识到三大云本质同构——理解思想比记服务名重要。**

**English**:
1. **BigQuery pushes "serverless" to the extreme: you don't even see a cluster**: compared to node-configured Redshift or resource-managed Spark, BigQuery's experience is — you write one SQL and PB-scale results return in seconds, **with no idea (and no need to know) how many machines were mobilized**. This is the ultimate storage-compute separation + columnar: data rests in storage, and when a query arrives it temporarily summons massive compute to scan in parallel, releasing after. For data scientists, this means **you focus on SQL and analysis, not ops** — the best embodiment of the cloud's "no ops" value. Snowflake (a later section) is similar; they represent the modern form of the warehouse.
2. **"Billing per bytes scanned" is double-edged — understand it to save, ignore it and the bill explodes**: our real measurement gives a striking number — for "total revenue on a given day," a `SELECT *` (scanning the whole 162MB table) costs **60x more** than `SELECT sum(revenue) WHERE date=...` (scanning 2.7MB), while both give **identical correct results**. If your query runs thousands of times a day on TB-scale tables, this difference is thousands of dollars vs tens per month. Key understanding: ① **you pay for scanned columns** — that huge `raw_payload` column you never look at, `SELECT *` scans it all; ② **you pay for scanned partitions** — no partition filter means a full-table scan; ③ **the most counterintuitive trap: `LIMIT 10` doesn't save money** — it still full-scans, just returns 10 rows at the end (billing is per scan, not per return). These aren't "optimization tricks" but the **basic literacy** of using BigQuery.
3. **Honest limits: serverless is great, but be clear on its tradeoffs and fit**. ① **Poor cost predictability**: pay-per-use means one mistyped query (forgetting a partition filter, or an exploding join) can instantly scan tens of TB and burn a fortune — so production needs **cost controls** (query byte limits, budget alerts, capacity pricing to lock cost). ② **Not all workloads fit**: BigQuery is optimized for **large-scale analytical scans (OLAP)**, not high-frequency small transactions/low-latency point lookups (that's an OLTP database or Bigtable's job); frequent small queries may also be uneconomical under pay-per-use. ③ **Vendor lock-in**: deep use of BigQuery's proprietary features (BQ ML, proprietary functions) binds you to GCP with high migration cost — so many teams insist on **standard SQL + open formats (also store a Parquet/Iceberg copy in GCS)** to keep portability. ④ **The three clouds are highly isomorphic**: GCS≈S3, BigQuery≈Athena/Redshift/Snowflake, Vertex AI≈SageMaker, Pub/Sub≈Kafka — **learn one cloud's mental model and switching clouds is just reading docs and changing APIs**; don't be intimidated by vendor marketing names. **Conclusion: BigQuery represents the ultimate serverless warehouse — storage-compute separation lets you query PB in seconds with no ops; but it bills per bytes scanned, and the iron rule to save money is partition filters + selecting only needed columns (never SELECT *, don't expect LIMIT to save), where one oversight can burn tens of times the cost; set cost guardrails, beware lock-in, and recognize the three clouds are fundamentally isomorphic — understanding ideas matters more than memorizing service names.**

> 💼 **实战视角 / Practical angle**
> **中文**:GCP 数据科学落地:①**数仓用 BigQuery**——省钱铁律:分区表(按日期)+ 分区过滤 + 只选需要列, 别 SELECT *, 用 `--dry_run` 预估扫描量, 设查询字节上限和预算告警;②**存储用 GCS**(存 Parquet, 也做数据湖);③**ML 用 Vertex AI**(训练/部署/pipeline)或直接 BQ ML(SQL 里训简单模型);④**流批用 Dataflow**(Apache Beam, 一套代码流批统一);⑤成本大时考虑**容量定价(slots)** 锁定 vs 按需;⑥物化视图/BI Engine 缓存高频查询。**跨云认知**:GCS=S3、BigQuery=Athena/Redshift、Vertex=SageMaker、Pub/Sub=Kafka、Dataproc=EMR。面试金句:*"BigQuery 无服务器数仓靠存算分离+列式秒查 PB, 按扫描字节计费; 省钱靠分区过滤和只选需要列(别 SELECT *、LIMIT 不减扫描), 用 dry_run 预估、设字节上限防账单爆炸; GCP 和 AWS 高度同构(GCS=S3/Vertex=SageMaker), 理解存算分离和按量计费的思想比记服务名重要。"*
> **English**: GCP data-science in practice: ① **warehouse via BigQuery** — cost rules: partitioned tables (by date) + partition filters + select only needed columns, never SELECT *, use `--dry_run` to estimate scan, set query byte limits and budget alerts; ② **storage via GCS** (store Parquet, also as a data lake); ③ **ML via Vertex AI** (training/deployment/pipelines) or BQ ML directly (train simple models in SQL); ④ **stream/batch via Dataflow** (Apache Beam, one codebase for both); ⑤ at high cost, consider **capacity pricing (slots)** to lock vs on-demand; ⑥ materialized views/BI Engine to cache frequent queries. **Cross-cloud mapping**: GCS=S3, BigQuery=Athena/Redshift, Vertex=SageMaker, Pub/Sub=Kafka, Dataproc=EMR. Interview line: *"BigQuery is a serverless warehouse using storage-compute separation + columnar to query PB in seconds, billed per bytes scanned; save money via partition filters and selecting only needed columns (never SELECT *, LIMIT doesn't cut the scan), use dry_run to estimate and byte limits to prevent bill explosions; GCP and AWS are highly isomorphic (GCS=S3, Vertex=SageMaker), so understanding storage-compute separation and pay-per-use matters more than memorizing service names."*

---
### 小结 / Summary
- **中文**:BigQuery=无服务器数仓(存算分离+列式, 不管集群秒查 PB); 核心是按扫描字节计费(成本∝扫描量, 与返回行数无关)。
- **English**: BigQuery = serverless warehouse (storage-compute separation + columnar, no clusters, PB in seconds); core is billing per bytes scanned (cost ∝ scan, independent of rows returned).
- **中文**:省钱铁律:分区+分区过滤、只选需要列(别 SELECT *、LIMIT 不减扫描)——真实测量差 60 倍。
- **English**: Cost rules: partition + partition filters, select only needed columns (never SELECT *, LIMIT doesn't cut the scan) — a measured 60x difference.
- **中文**:GCP 与 AWS 高度同构(GCS=S3/Vertex=SageMaker/Pub-Sub=Kafka); 理解思想比记服务名重要, 警惕锁定和账单爆炸。
- **English**: GCP ≈ AWS (GCS=S3, Vertex=SageMaker, Pub/Sub=Kafka); understanding ideas beats memorizing names; beware lock-in and bill explosions.
